# TSTR Evaluation


In [18]:
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({'legend.fontsize': 9,
                     'axes.titlesize': 10})

In [19]:
RESULT_DIR = 'results/tstr'
CLASSES = ['bradypnea', 'eupnea', 'tachypnea']
MODEL_ORDER = ['trtr', 'cvae', 'vae', 'gan']
REGION_ORDER = ['mouth', 'nose']
COLORS = {'trtr': 'tab:blue', 'cvae': 'tab:green', 'vae': 'tab:orange', 'gan': 'tab:red'}
REGION_STYLE = {'mouth': '-', 'nose': '--'}
models = ['cvae', 'vae', 'gan']

# load json files
records = []
for path in glob.glob(f'{RESULT_DIR}/*.json'):
    with open(path) as f:
        records.append(json.load(f))

# load summary
df = pd.read_csv(f'{RESULT_DIR}/summary.csv')
df['model'] = pd.Categorical(df['model'], categories=MODEL_ORDER, ordered=True)
df = df.sort_values(['region', 'model', 'seed']).reset_index(drop=True)

print(df.to_string(index=False))

model region  seed  accuracy  f1_weighted  roc_auc_ovr  log_loss  f1_bradypnea  f1_eupnea  f1_tachypnea  feature_overlap  n_synthetic
 trtr  mouth     7  0.928571     0.930153     1.000000  0.196067      0.900000   0.941176      0.947368              NaN          215
 trtr  mouth    42  0.928571     0.925897     1.000000  0.225687      0.875000   0.947368      0.952381              NaN          215
 trtr  mouth   123  1.000000     1.000000     1.000000  0.109828      1.000000   1.000000      1.000000              NaN          215
 cvae  mouth     7  0.857143     0.858814     0.971345  0.347247      0.842105   0.842105      0.888889              0.3          215
 cvae  mouth    42  0.821429     0.817266     0.908285  0.555677      0.947368   0.761905      0.750000              0.3          215
 cvae  mouth   123  0.892857     0.892063     0.961209  0.435483      0.875000   0.842105      0.952381              0.4          215
  vae  mouth     7  0.321429     0.325081     0.542008  1.0988

In [24]:
# average summary
metrics_cols = ['accuracy', 'f1_weighted', 'roc_auc_ovr', 'log_loss', 'f1_bradypnea', 'f1_eupnea', 'f1_tachypnea', 'feature_overlap']
agg = df.groupby(['model', 'region'], observed=True)[metrics_cols].agg(['mean', 'std']).round(3)
rows = []
for (model, region), g in df.groupby(['model', 'region'], observed=True):
    row = {'model': model, 'region': region}
    for m in metrics_cols:
        mu = g[m].mean()
        sd = g[m].std()
        if pd.isna(sd) or len(g) < 2:
            row[m] = f'{mu:.3f}'
        else:
            row[m] = f'{mu:.3f} ± {sd:.3f}'
    rows.append(row)

summary = pd.DataFrame(rows)
summary['model'] = pd.Categorical(summary['model'], categories=MODEL_ORDER, ordered=True)
summary = summary.sort_values(['region', 'model']).reset_index(drop=True)
print(summary.to_string(index=False))

model region      accuracy   f1_weighted   roc_auc_ovr      log_loss  f1_bradypnea     f1_eupnea  f1_tachypnea feature_overlap
 trtr  mouth 0.952 ± 0.041 0.952 ± 0.042 1.000 ± 0.000 0.177 ± 0.060 0.925 ± 0.066 0.963 ± 0.032 0.967 ± 0.029             nan
 cvae  mouth 0.857 ± 0.036 0.856 ± 0.037 0.947 ± 0.034 0.446 ± 0.105 0.888 ± 0.054 0.815 ± 0.046 0.864 ± 0.104   0.333 ± 0.058
  vae  mouth 0.405 ± 0.312 0.402 ± 0.314 0.552 ± 0.307 1.094 ± 0.124 0.426 ± 0.289 0.307 ± 0.295 0.465 ± 0.378   0.600 ± 0.000
 trtr   nose 0.772 ± 0.040 0.767 ± 0.042 0.923 ± 0.022 0.516 ± 0.072 0.869 ± 0.047 0.769 ± 0.067 0.655 ± 0.111             nan
 cvae   nose 0.588 ± 0.040 0.583 ± 0.055 0.711 ± 0.035 1.035 ± 0.094 0.624 ± 0.066 0.559 ± 0.225 0.563 ± 0.117   0.433 ± 0.115
  vae   nose 0.263 ± 0.091 0.254 ± 0.086 0.411 ± 0.074 1.152 ± 0.051 0.215 ± 0.158 0.366 ± 0.088 0.175 ± 0.161   0.533 ± 0.058


In [ ]:
metrics_main = ['accuracy', 'f1_weighted', 'roc_auc_ovr', 'log_loss']
metric_labels = ['Accuracy', 'F1 (weighted)', 'ROC-AUC (OvR)', 'Log Loss']

rows = []
for region in REGION_ORDER:
    trtr_row = df[(df.model == 'trtr') & (df.region == region)].iloc[0]
    for model in models:
        r = df[(df.model == model) & (df.region == region)].iloc[0]
        entry = {'model': model, 'region': region}
        for m in metrics_main:
            t = trtr_row[m]
            v = r[m]
            if m == 'log_loss':
                entry[m] = t / v if v > 0 else np.nan
            else:
                entry[m] = v / t if t > 0 else np.nan
        rows.append(entry)

util_df = pd.DataFrame(rows)
fig, axes = plt.subplots(1, len(metrics_main), figsize=(14, 4), sharey=True)
x = np.arange(2)
width = 0.15
for ax, m, label in zip(axes, metrics_main, metric_labels):
    for i, model in enumerate(models):
        vals = [util_df[(util_df.model == model) & (util_df.region == r)][m].values[0] for r in REGION_ORDER]
        bars = ax.bar(x + i * width, vals, width, label=model.upper(), color=COLORS[model], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,f'{v:.2f}', ha='center', va='bottom', fontsize=8)
    ax.axhline(1.0, linewidth=1, label='TRTR baseline')
    ax.set_title(label)
    ax.set_xticks(x + width)
    ax.set_xticklabels(REGION_ORDER)
    ax.set_ylim(0, 1.25)
    ax.set_ylabel('Utility score' if label == metric_labels[0] else '')
    ax.legend(loc='upper right') if m == metrics_main[3] else None
    ax.grid()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)
axes = axes.flatten()
metrics_abs = ['accuracy', 'f1_weighted', 'roc_auc_ovr', 'log_loss']
metric_labels_abs = ['Accuracy', 'F1 (weighted)', 'ROC-AUC (OvR)', 'Log Loss ↓']
invert = [False, False, False, True]
x = np.arange(len(MODEL_ORDER))
width = 0.35
hatch = {'mouth': '', 'nose': '//'}
for ax, m, label, inv in zip(axes, metrics_abs, metric_labels_abs, invert):
    for j, region in enumerate(REGION_ORDER):
        vals = [df[(df.model == mo) & (df.region == region)][m].values[0] for mo in MODEL_ORDER]
        bars = ax.bar(x + j * width, vals, width, label=region, color=[COLORS[mo] for mo in MODEL_ORDER], hatch=hatch[region], alpha=0.85, edgecolor='white')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005 * (ax.get_ylim()[1] if ax.get_ylim()[1] else 1), f'{v:.2f}', ha='center', va='bottom', fontsize=7.5, rotation=0)
    ax.set_title(label)
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels([m.upper() for m in MODEL_ORDER])
    legend_patches = [mpatches.Patch(facecolor='silver', edgecolor='black', hatch=hatch[r], label=r) for r in REGION_ORDER]
    ax.legend(handles=legend_patches, loc='upper left') if m == metrics_abs[3] else None

model_patches = [mpatches.Patch(facecolor=COLORS[m], label=m.upper()) for m in MODEL_ORDER]
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, region in zip(axes, REGION_ORDER):
    matrix = []
    row_labels = []
    for model in MODEL_ORDER:
        r = df[(df.model == model) & (df.region == region)].iloc[0]
        matrix.append([r['f1_bradypnea'], r['f1_eupnea'], r['f1_tachypnea']])
        row_labels.append(model.upper())

    mat = np.array(matrix)
    im = ax.imshow(mat, vmin=0, vmax=1, cmap='viridis', aspect='equal')

    ax.set_xticks(range(len(CLASSES)))
    ax.set_xticklabels(CLASSES)
    ax.set_yticks(range(len(MODEL_ORDER)))
    ax.set_yticklabels(row_labels)
    ax.set_title(f'Region: {region}')

    for i in range(len(MODEL_ORDER)):
        for j in range(len(CLASSES)):
            v = mat[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8, color='white' if v < 0.5 else 'black')

plt.colorbar(im, ax=axes[-1], label='F1 score')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(2)
width = 0.3
for i, model in enumerate(models):
    vals = [df[(df.model == model) & (df.region == r)]['feature_overlap'].values[0] for r in REGION_ORDER]
    bars = ax.bar(x + i * width, vals, width, label=model.upper(), color=COLORS[model], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,f'{v:.1%}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(REGION_ORDER)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Overlap')
ax.set_title('Top-10 SHAP Feature Overlap: Synthetic vs Real')
ax.legend()
plt.tight_layout()
plt.show()